In [11]:
import psycopg2
import os
from dotenv import load_dotenv
import csv
from datetime import datetime, timedelta

# Load environment variables from .env
load_dotenv()

# Database connection information
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

def extract_coordinates(coord_str):
    # Extract latitude and longitude from the coordinate string
    coord_str = coord_str.strip('()')
    lat, lon = map(float, coord_str.split(', '))
    return lat, lon

def get_latitudes_and_longitudes(provider_id, model_id):
    try:
        # Establish a connection to the PostgreSQL database
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USERNAME,
            password=DB_PASSWORD
        )

        # Create a cursor object
        cursor = conn.cursor()

        # Query the latitudes and longitudes based on provider_id and model_id from the lat_lon_schema table
        cursor.execute("SELECT latitudes, longitudes FROM lat_lon_schema WHERE provider_id = %s AND model_id = %s", (provider_id, model_id))

        # Fetch the row
        row = cursor.fetchone()

        if row:
            latitudes, longitudes = row
            return latitudes, longitudes
        else:
            print(f"No latitudes and longitudes found for provider_id {provider_id} and model_id {model_id}.")
            return None, None

    except Exception as e:
        print(f"Error: {e}")
        return None, None

def write_data_to_csv_with_coordinates(table_name, output_file, provider_id, model_id):
    try:
        # Establish a connection to the PostgreSQL database
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USERNAME,
            password=DB_PASSWORD
        )

        # Create a cursor object
        cursor = conn.cursor()

        # Query the data from the specified table
        cursor.execute(f"SELECT start_date, end_date, interval, data FROM {table_name}")

        # Fetch all rows
        rows = cursor.fetchall()

        if rows:
            # Prepare data for CSV
            csv_data = []
            latitudes, longitudes = get_latitudes_and_longitudes(provider_id, model_id)

            for row in rows:
                start_date, end_date, interval, data = row
                current_date = start_date
                interval_seconds = int(interval.total_seconds())

                for day_data, lat, lon in zip(data, latitudes, longitudes):
                    current_date += timedelta(seconds=interval_seconds)

                    # Replace None with 0.0 in weather data
                    day_data = [0.0 if value is None else value for value in day_data]

                    # Duplicate latitude and longitude once for each day's data
                    coordinates = [(lat, lon)] * len(day_data)

                    # Combine timestamp, weather data, and coordinates
                    combined_data = list(zip([current_date] * len(day_data), day_data, *zip(*coordinates)))

                    # Append each combined data point to the CSV data
                    csv_data.extend(combined_data)

            # Write data to CSV file
            with open(output_file, "w", newline="") as csv_file:
                csv_writer = csv.writer(csv_file)
                csv_writer.writerow(["Timestamp", "Weather Data", "Latitude", "Longitude"])  # CSV header
                csv_writer.writerows(csv_data)

            print(f"CSV file '{output_file}' created successfully.")
        else:
            print(f"No data found in table '{table_name}'.")

        # Close the cursor and the connection
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    # Specify the table name, output file name, provider_id, and model_id
    table_name = "total_precipitation_aladin"  # Replace with your table name
    output_file = "./data/output_with_coordinates.csv"  # Replace with your desired output file name
    provider_id = "6be8cea2-f29b-4198-aa68-10c57845ad25"  # Replace with the desired provider_id as a string
    model_id = "581e4233-dc8c-44d3-b351-c115dc32fc53"  # Replace with the desired model_id as a string

    write_data_to_csv_with_coordinates(table_name, output_file, provider_id, model_id)


CSV file './data/output_with_coordinates.csv' created successfully.
